# Dependencies & Setup Guide (LibKGE + CODEX-S + Winner Model)

This project depends on multiple datasets and preprocessed resources hosted on Kaggle and Hugging Face. Follow the instructions carefully to ensure smooth execution.

All datasets used in this project are publicly available.

---

## Required Kaggle Datasets

- CODEX Dataset  
  https://www.kaggle.com/datasets/aaryaupi/codex-dataset/versions/1  

- Held-Out Evaluation Set  
  https://www.kaggle.com/datasets/aaryaupi/held-out  

- CODEX-S Validation Dataset  
  https://www.kaggle.com/datasets/aaryaupi/codex-s-val-dataset  

- LibKGE Support Dataset  
  https://www.kaggle.com/datasets/aaryaupi/model-libkge-kaggle-upload/versions/2  

- Winner Model (Kaggle Version)  
  https://www.kaggle.com/datasets/aaryaupi/winner-model  

---

## Winner Model (Best Reproduced Results)

The state-of-the-art model (highest MRR and Hits@K) is also available on Hugging Face:

https://huggingface.co/aaryaupadhya20/codex-s-complex-winner  

- Pretrained ComplEx-based model  
- Ready for inference and evaluation  
- Recommended instead of retraining (saves ~5 hours)

---

## Important Execution Notes

- This project is designed to run on Kaggle notebooks.
- USE A T4 GPU for stable execution and reproducibility. P100 may fail or produce unstable results.
- Some file paths in the notebook are hard-coded for Kaggle environments and may need manual adjustment if running outside Kaggle.
- If datasets do not load automatically, use the Kaggle option **"Load Notebook with Datasets"** to ensure all inputs are correctly attached.

---

## Dataset Loading Warning

All datasets are publicly available, but they must be explicitly attached in Kaggle.

If datasets fail to load:
- Use **"Load Notebook" → "Add Data" option in Kaggle**
- Ensure all linked datasets above are attached to the session

---

## File Paths (May Require Update)

Example hardcoded paths used in the notebook:

```python
HELD_OUT_INPUT = "/kaggle/input/datasets/aaryaupi/codex-s-val-dataset/CODEX_S_held_out (2).json"
HELD_OUT_OUT   = "/kaggle/working/codex_bert_held_out.json"

In [1]:
!pip install pykeen huggingface_hub -q
!pip install -q langchain langchain-community langchain-core \
     langchain-huggingface faiss-cpu sentence-transformers \
     transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not

In [2]:
import warnings
warnings.filterwarnings("ignore")

import os, json, csv, time, pickle, torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from collections import defaultdict, deque, Counter
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
import torch, pickle 

In [3]:
import json, os

BASE = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master"

# Peek at entities.json structure
with open(f"{BASE}/data/entities/en/entities.json") as f:
    ent_json = json.load(f)

# See what it looks like
first_keys = list(ent_json.keys())[:3]
for k in first_keys:
    print(k, "→", ent_json[k])

# Peek at one txt file
sample_txt = f"{BASE}/data/entities/en/extracts/extracts/Q100.txt"
with open(sample_txt) as f:
    print("\nQ100.txt:\n", f.read()[:300])

Q202042 → {'wiki': 'https://en.wikipedia.org/wiki/Euskaltzaindia', 'description': 'official academic language regulatory institution which watches over the Basque language', 'label': 'Euskaltzaindia'}
Q113400 → {'wiki': 'https://en.wikipedia.org/wiki/Fritz_Glatz', 'description': 'Austrian racing driver', 'label': 'Fritz Glatz'}
Q432602 → {'wiki': 'https://en.wikipedia.org/wiki/David_Hidalgo', 'description': 'American musician', 'label': 'David Hidalgo'}

Q100.txt:
 Boston (UK: , US: ) is the capital and most populous city of the Commonwealth of Massachusetts in the United States, and the 21st most populous city in the United States. The city proper covers 49 square miles (127 km2) with an estimated population of 694,583 in 2018, also making it the most populou


In [4]:
import json, os, pandas as pd

BASE = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master"


with open(f"{BASE}/data/entities/en/entities.json") as f:
    ent_json = json.load(f)   # { "Q202042": {"label": "Euskaltzaindia", ...}, ... }

with open(f"{BASE}/data/relations/en/relations.json") as f:
    rel_json = json.load(f)   # same nested structure for relations

def ent(qid): return ent_json.get(qid, {}).get("label", qid)
def rel(pid): return rel_json.get(pid, {}).get("label", pid)

def load_split(split_name):
    path = f"{BASE}/data/triples/codex-s/{split_name}.txt"
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 3:
                h, r, t = parts
                rows.append((ent(h), rel(r), ent(t)))
    return pd.DataFrame(rows, columns=["head", "relation", "tail"])

df_train = load_split("train")
df_test  = load_split("test")
df_valid = load_split("valid")

print(f"Train: {len(df_train)}  Valid: {len(df_valid)}  Test: {len(df_test)}")
print(df_train.head(10))

def read_id_list(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

entity_qids   = sorted(ent_json.keys())
relation_pids = sorted(rel_json.keys())

entity_to_id   = {ent(qid): i for i, qid in enumerate(entity_qids)}
relation_to_id = {rel(pid): i for i, pid in enumerate(relation_pids)}
id_to_entity   = {i: ent(qid) for i, qid in enumerate(entity_qids)}
id_to_relation = {i: rel(pid) for i, pid in enumerate(relation_pids)}

print(f"Entities: {len(entity_to_id)}   Relations: {len(relation_to_id)}")
print(id_to_entity[0], "|", id_to_entity[1], "|", id_to_entity[2])
print(id_to_relation[0], "|", id_to_relation[1])

Train: 32888  Valid: 1827  Test: 1828
                           head                              relation  \
0                Leonhard Euler  languages spoken, written, or signed   
1                 Carl Djerassi                        cause of death   
2                      Colombia                             member of   
3  Aleksey Nikolayevich Tolstoy  languages spoken, written, or signed   
4                   Netherlands                             member of   
5                 Henry Rollins                            occupation   
6                Richard Wagner                            occupation   
7                   Jaden Smith                            occupation   
8               La Toya Jackson                               sibling   
9         John Cameron Mitchell                            occupation   

                                tail  
0                             German  
1                             cancer  
2  International Finance Corporation  
3 

In [5]:
df_train.to_csv("CODEX_S_train.csv",index = None)
df_test.to_csv("CODEX_S_test.csv",index = None)
df_valid.to_csv("CODEX_S_valid.csv",index = None)

# Load The Pretrained CompLEX model from The Private Kaggle dataset 

In [6]:
!git clone https://github.com/uma-pi1/kge.git

Cloning into 'kge'...
remote: Enumerating objects: 7785, done.
remote: Counting objects: 100% (1828/1828), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 7785 (delta 1728), reused 1697 (delta 1697), pack-reused 5957 (from 1)
Receiving objects: 100% (7785/7785), 1.86 MiB | 8.36 MiB/s, done.
Resolving deltas: 100% (5887/5887), done.


In [7]:
import sys

# CLEAN everything related to kge
sys.modules.pop("kge", None)

# REMOVE wrong paths
sys.path = [p for p in sys.path if "kge" not in p]

# ADD ONLY THIS
sys.path.insert(0, "/kaggle/working/kge")

In [8]:
import kge
print(kge.__file__)

/kaggle/working/kge/kge/__init__.py


In [9]:
import sys
sys.modules.pop("kge", None)

<module 'kge' from '/kaggle/working/kge/kge/__init__.py'>

In [10]:
file_path = "/kaggle/working/kge/kge/job/__init__.py"

with open(file_path, "r") as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if "search_ax" in line:
        continue  # REMOVE the line entirely
    new_lines.append(line)

with open(file_path, "w") as f:
    f.writelines(new_lines)

print("Removed Ax import completely")

Removed Ax import completely


In [11]:
!rm /kaggle/working/kge/kge/job/search_ax.py

In [12]:
!ls /kaggle/working/kge/kge

cli.py		     config.py	 indexing.py  job	   misc.py  __pycache__
config-default.yaml  dataset.py  __init__.py  __main__.py  model    util


In [13]:
file_path = "/kaggle/working/kge/kge/job/__init__.py"

with open(file_path, "r") as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if "search_grash" in line:
        continue  # remove this import
    new_lines.append(line)

with open(file_path, "w") as f:
    f.writelines(new_lines)

print("Removed GraSH import")

Removed GraSH import


In [14]:
from kge.model import KgeModel
from kge.util.io import load_checkpoint

print("Import worked")

Import worked


In [15]:
import sys
print(sys.path)

['/kaggle/working/kge', '/kaggle/working', '/kaggle/lib/kagglegym', '/kaggle/lib', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']


In [16]:
import os
print(os.path.exists("/kaggle/working/kge/kge/model"))

True


In [17]:
from kge.model import KgeModel
from kge.util.io import load_checkpoint

print("Import worked")

Import worked


In [18]:
MODEL_PATH = r"/kaggle/input/datasets/aaryaupi/winner-model/codex-s-complex-winner/winner_model.pt"
print(MODEL_PATH)


/kaggle/input/datasets/aaryaupi/winner-model/codex-s-complex-winner/winner_model.pt


In [19]:
file_path = "/kaggle/working/kge/kge/util/io.py"

with open(file_path, "r") as f:
    content = f.read()

content = content.replace(
    "torch.load(checkpoint_file, map_location=\"cpu\")",
    "torch.load(checkpoint_file, map_location=\"cpu\", weights_only=False)"
)

with open(file_path, "w") as f:
    f.write(content)

print("Patched torch.load")

Patched torch.load


In [20]:
!ls /kaggle/input/datasets/aaryaupi/codex-dataset/codex-master/data/triples

codex-l  codex-m  codex-s  raw


In [21]:
import os

os.environ["KGE_DATASET_DIR"] = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master/data/triples"

In [22]:
import torch
from kge.config import Config

torch.serialization.add_safe_globals([
    torch.torch_version.TorchVersion,
    Config
])

In [23]:
import sys
import torch

sys.path.insert(0, "/kaggle/working")

from kge.config import Config
from kge.util.io import load_checkpoint

# PyTorch safety fix
torch.serialization.add_safe_globals([
    torch.torch_version.TorchVersion,
    Config
])

# reload checkpoint
checkpoint = load_checkpoint(MODEL_PATH, device="cpu")

print("Checkpoint loaded")

Checkpoint loaded ✅


In [24]:
from pykeen.triples import TriplesFactory

In [25]:
import torch, json, pickle, numpy as np
from pathlib import Path

MODEL_DIR  = "/kaggle/input/datasets/aaryaupi/model-libkge-kaggle-upload/kaggle_upload"
CODEX_BASE = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master"

CKPT_PATH     = f"{MODEL_DIR}/winner_model.pt"
ENTITY_IDS    = f"{MODEL_DIR}/entity_ids.del"
RELATION_IDS  = f"{MODEL_DIR}/relation_ids.del"

import torch

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
state_dict = ckpt["model"][0]

entity_emb   = state_dict["_base_model._entity_embedder._embeddings.weight"].detach().float()
relation_emb = state_dict["_base_model._relation_embedder._embeddings.weight"].detach().float()

print(f"entity_emb:   {entity_emb.shape}")
print(f"relation_emb: {relation_emb.shape}")
print(f"trained epoch: {ckpt['epoch']}")

# Fix here
best_valid = ckpt["valid_trace"][-1]
print(f"best valid MRR: {best_valid['mean_reciprocal_rank_filtered_with_test']:.4f}")

entity_emb:   torch.Size([2034, 512])
relation_emb: torch.Size([84, 512])
trained epoch: 295
best valid MRR: 0.4742


In [26]:

def load_del(path):
    id_to_qid = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                id_to_qid[int(parts[0])] = parts[1]
    return id_to_qid

id_to_entity_qid   = load_del(ENTITY_IDS)
id_to_relation_pid = load_del(RELATION_IDS)
entity_qid_to_id   = {v: k for k, v in id_to_entity_qid.items()}
relation_pid_to_id = {v: k for k, v in id_to_relation_pid.items()}


with open(f"{CODEX_BASE}/data/entities/en/entities.json")  as f: ent_json = json.load(f)
with open(f"{CODEX_BASE}/data/relations/en/relations.json") as f: rel_json = json.load(f)

def elabel(qid): return ent_json.get(qid, {}).get("label", qid)
def rlabel(pid): return rel_json.get(pid, {}).get("label", pid)

id_to_entity_label   = {i: elabel(q) for i, q in id_to_entity_qid.items()}
id_to_relation_label = {i: rlabel(p) for i, p in id_to_relation_pid.items()}
label_to_entity_id   = {v: k for k, v in id_to_entity_label.items()}
label_to_relation_id = {v: k for k, v in id_to_relation_label.items()}

print(f"Entities:  {len(id_to_entity_label)}")
print(f"Relations: {len(id_to_relation_label)}")
print(f"\nAll relations:")
for i, label in sorted(id_to_relation_label.items()):
    print(f"  {i:3d}: {label}")

Entities:  2034
Relations: 42

All relations:
    0: languages spoken, written, or signed
    1: cause of death
    2: member of
    3: occupation
    4: sibling
    5: diplomatic relation
    6: continent
    7: ethnic group
    8: country of citizenship
    9: employer
   10: educated at
   11: place of birth
   12: influenced by
   13: field of work
   14: genre
   15: record label
   16: instrument
   17: official language
   18: unmarried partner
   19: place of death
   20: country
   21: religion
   22: movement
   23: residence
   24: spouse
   25: member of political party
   26: place of burial
   27: part of
   28: medical condition
   29: child
   30: time period
   31: headquarters location
   32: narrative location
   33: location of formation
   34: head of state
   35: named after
   36: parent organization
   37: notable works
   38: cast member
   39: founded by
   40: country of origin
   41: practiced by


In [27]:
def kvs_to_dict(kvs_index):
    index_of_key = kvs_index._index_of_key   # dict: key -> row_idx
    values        = kvs_index._values         # 1D LongTensor, all values concatenated
    offsets        = kvs_index._values_offset  # 1D LongTensor, len = n_keys + 1

    result = {}
    for key, row_idx in index_of_key.items():
        start = offsets[row_idx].item()
        end   = offsets[row_idx + 1].item()
        result[key] = set(values[start:end].tolist())
    return result

with open(f"{MODEL_DIR}/index-train_sp_to_o.pckl", "rb") as f:
    train_sp_to_o = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-train_po_to_s.pckl", "rb") as f:
    train_po_to_s = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_sp_to_o.pckl", "rb") as f:
    test_sp_to_o  = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_po_to_s.pckl", "rb") as f:
    test_po_to_s  = kvs_to_dict(pickle.load(f))

print("Filter indexes loaded")

Filter indexes loaded


In [28]:
with open(f"{MODEL_DIR}/index-train_sp_to_o.pckl", "rb") as f:
    idx = pickle.load(f)

# Inspect what's available
print(type(idx))
print([a for a in dir(idx) if not a.startswith("__")])

<class 'kge.indexing.KvsAllIndex'>
['_convert_key_to_correct_type', '_create_index_of_key_dict', '_get_all_impl', '_index_of_key', '_keys', '_values', '_values_of', '_values_offset', 'default_factory', 'get', 'get_all', 'items', 'key_cols', 'keys', 'sort_triples_by_keys', 'value_col', 'values']


In [29]:
DIM = entity_emb.shape[1] // 2  # 256

def _re(e): return e[..., :DIM]
def _im(e): return e[..., DIM:]

def score_triple(s, p, o):
    sr, si = _re(entity_emb[s]), _im(entity_emb[s])
    pr, pi = _re(relation_emb[p]), _im(relation_emb[p])
    or_, oi = _re(entity_emb[o]), _im(entity_emb[o])
    return (torch.sum(sr*pr*or_) - torch.sum(si*pi*or_) +
            torch.sum(sr*pi*oi) + torch.sum(si*pr*oi)).item()

def _scores_sp(s, p):
    sr, si = _re(entity_emb[s]), _im(entity_emb[s])
    pr, pi = _re(relation_emb[p]), _im(relation_emb[p])
    return (_re(entity_emb) @ (sr*pr - si*pi) +
            _im(entity_emb) @ (sr*pi + si*pr))

def _scores_po(p, o):
    pr, pi = _re(relation_emb[p]), _im(relation_emb[p])
    or_, oi = _re(entity_emb[o]), _im(entity_emb[o])
    return (_re(entity_emb) @ (pr*or_ - pi*oi) +
            _im(entity_emb) @ (pr*oi + pi*or_))


def kvs_to_dict(kvs_index):
    index_of_key = kvs_index._index_of_key
    values       = kvs_index._values
    offsets      = kvs_index._values_offset
    result = {}
    for key, row_idx in index_of_key.items():
        start = offsets[row_idx].item()
        end   = offsets[row_idx + 1].item()
        result[key] = set(values[start:end].tolist())
    return result

with open(f"{MODEL_DIR}/index-train_sp_to_o.pckl", "rb") as f: train_sp_to_o = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-train_po_to_s.pckl", "rb") as f: train_po_to_s = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_sp_to_o.pckl",  "rb") as f: test_sp_to_o  = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_po_to_s.pckl",  "rb") as f: test_po_to_s  = kvs_to_dict(pickle.load(f))
print("Filter indexes loaded")


def evaluate(split="test"):
    path = f"{CODEX_BASE}/data/triples/codex-s/{split}.txt"
    rr_list, hits1_list, hits10_list = [], [], []

    with open(path) as f:
        triples = [l.strip().split("\t") for l in f if l.strip()]

    for h_qid, r_pid, t_qid in triples:
        s = entity_qid_to_id.get(h_qid)
        p = relation_pid_to_id.get(r_pid)
        o = entity_qid_to_id.get(t_qid)
        if None in (s, p, o):
            continue

        scores = _scores_sp(s, p).clone()
        known  = train_sp_to_o.get((s, p), set()) | test_sp_to_o.get((s, p), set())
        known.discard(o)
        for k in known:
            scores[k] = float("-inf")

        rank = (scores > scores[o]).sum().item() + 1
        rr_list.append(1.0 / rank)
        hits1_list.append(1 if rank == 1 else 0)
        hits10_list.append(1 if rank <= 10 else 0)

    mrr    = float(np.mean(rr_list))
    hits1  = float(np.mean(hits1_list))
    hits10 = float(np.mean(hits10_list))
    print(f"\n── {split.upper()} SET ──────────────────────────────")
    print(f"  MRR:     {mrr:.4f}  (paper: 0.465)")
    print(f"  Hits@1:  {hits1:.4f}  (paper: 0.372)")
    print(f"  Hits@10: {hits10:.4f}  (paper: 0.646)")
    return mrr, hits1, hits10

evaluate("test")

Filter indexes loaded

── TEST SET ──────────────────────────────
  MRR:     0.5759  (paper: 0.465)
  Hits@1:  0.4294  (paper: 0.372)
  Hits@10: 0.8714  (paper: 0.646)


(0.575868337490121, 0.42943107221006566, 0.8714442013129103)

In [30]:
def build_graph_from_df(df):
    graph = defaultdict(list)
    for _, row in df.iterrows():
        graph[row["head"]].append((row["relation"], row["tail"]))
    return graph

graph = build_graph_from_df(df_train)
print(f"Graph nodes: {len(graph)}")

Graph nodes: 1702


In [31]:
EMBS_CACHE = None

def get_entity_embeddings():
    """Return real-valued entity embedding matrix for similarity."""
    embs = entity_emb.float()
    # ComplEx: collapse complex dims by taking magnitude per pair
    re = embs[:, :DIM]
    im = embs[:, DIM:]
    return torch.sqrt(re**2 + im**2)   # shape: (n_entities, DIM)

# ── Subgraph helpers ──────────────────────────────────────────
def extract_subgraph(entity, graph, hops=2, max_triples=150):
    subgraph = []
    visited  = {entity}
    queue    = deque([(entity, 0)])
    while queue and len(subgraph) < max_triples:
        node, depth = queue.popleft()
        if depth >= hops:
            continue
        for rel_label, tail in graph.get(node, []):
            if len(subgraph) >= max_triples:
                break
            subgraph.append((node, rel_label, tail))
            if tail not in visited:
                visited.add(tail)
                queue.append((tail, depth + 1))
    return subgraph

def focused_subgraph(entities, graph, hops=2, max_triples=100, query_relation=None):
    entity_set  = set(entities)
    all_triples = []
    seen        = set()
    for ent_node in entities:
        for triple in extract_subgraph(ent_node, graph, hops, max_triples):
            if triple not in seen:
                seen.add(triple)
                all_triples.append(triple)
    if len(all_triples) <= max_triples:
        return all_triples
    tier1, tier2, tier3, tier4 = [], [], [], []
    for h, r, t in all_triples:
        if query_relation and r == query_relation:       tier1.append((h, r, t))
        elif h in entity_set and t in entity_set:        tier2.append((h, r, t))
        elif h in entity_set or  t in entity_set:        tier3.append((h, r, t))
        else:                                             tier4.append((h, r, t))
    return (tier1 + tier2 + tier3 + tier4)[:max_triples]

def hop_classifier(head, tail, graph, target_relation=None):
    for relation, t in graph.get(head, []):
        if t == tail:
            if target_relation is None or relation == target_relation:
                return "single", 1, relation
    direct_wrong = [r for r, t in graph.get(head, []) if t == tail]
    if direct_wrong:
        return "multi", 1.5, f"direct but via {direct_wrong[0]}"
    paths_found = []
    for r1, mid in graph.get(head, []):
        for r2, t2 in graph.get(mid, []):
            if t2 == tail:
                paths_found.append(f"{head}-{r1}->{mid}-{r2}->{tail}")
    if paths_found:
        return "multi", 2, paths_found[0]
    return "none", 99, []

In [32]:
def get_entity_relations(entity, df):
    head_rels = set(df[df["head"] == entity]["relation"])
    tail_rels = set(df[df["tail"] == entity]["relation"])
    return head_rels | tail_rels

def similarity_summary(entity, k=5):
    e_id  = label_to_entity_id[entity]
    e_vec = EMBS_CACHE[e_id]
    sims  = F.cosine_similarity(e_vec.unsqueeze(0), EMBS_CACHE).detach().cpu().numpy()
    ranked = np.argsort(sims)[::-1]
    entity_rels = get_entity_relations(entity, df_train)
    results = []
    for idx in ranked:
        name = id_to_entity_label[idx]
        if name == entity:
            continue
        score = sims[idx]
        neighbor_rels = get_entity_relations(name, df_train)
        shared    = len(entity_rels & neighbor_rels)
        total     = len(entity_rels | neighbor_rels)
        rel_score = shared / total if total > 0 else 0.0
        results.append((name, score, shared, rel_score))
        if len(results) == k:
            break
    parts   = [f"{n}(sim={s:.2f},shared={sh},rel={rs:.2f})"
               for n, s, sh, rs in results]
    summary = f"{entity} most similar to: {', '.join(parts)}"
    return summary, results

In [33]:
def get_full_ranking_filtered_batched(head, relation, true_tail):
    """
    Score ALL tail candidates using raw ComplEx embeddings.
    Replaces winner_model.score_hrt() entirely.
    """
    s = label_to_entity_id.get(head)
    p = label_to_relation_id.get(relation)
    o = label_to_entity_id.get(true_tail)

    if None in (s, p, o):
        raise ValueError(f"Unknown entity/relation: {head}, {relation}, {true_tail}")

    with torch.no_grad():
        all_scores = _scores_sp(s, p).cpu()   # shape: (n_entities,)

    # exclude head itself from ranking
    head_id = label_to_entity_id[head]
    all_scores[head_id] = float("-inf")

    scores_list = all_scores.tolist()
    ranked = sorted(
        [(id_to_entity_label[i], scores_list[i]) for i in range(len(scores_list))
         if scores_list[i] != float("-inf")],
        key=lambda x: -x[1]
    )

    ranked_entities = [e for e, _ in ranked]
    true_rank       = ranked_entities.index(true_tail) + 1

    return {
        "predicted":       ranked[0][0],
        "predicted_score": ranked[0][1],
        "true_tail":       true_tail,
        "true_score":      scores_list[o],
        "true_rank":       true_rank,
        "full_ranking":    [(e, round(sc, 3)) for e, sc in ranked],
    }


In [34]:
def build_type_constraints(df):
    rel_to_tail_counts = defaultdict(lambda: defaultdict(int))
    rel_to_head_counts = defaultdict(lambda: defaultdict(int))
    rel_to_pair_counts = defaultdict(lambda: defaultdict(int))
    for _, row in df.iterrows():
        h, r, t = row["head"], row["relation"], row["tail"]
        rel_to_tail_counts[r][t] += 1
        rel_to_head_counts[r][h] += 1
        rel_to_pair_counts[r][(h, t)] += 1
    rel_head_to_ranked_tails = defaultdict(dict)
    for r, pair_counts in rel_to_pair_counts.items():
        head_to_tails = defaultdict(dict)
        for (h, t), count in pair_counts.items():
            head_to_tails[h][t] = count
        for h, tail_counts in head_to_tails.items():
            total = sum(tail_counts.values())
            rel_head_to_ranked_tails[(r, h)] = {
                t: round(c / total, 4)
                for t, c in sorted(tail_counts.items(), key=lambda x: -x[1])
            }
    rel_to_tail_dist = {}
    for r, tail_counts in rel_to_tail_counts.items():
        total = sum(tail_counts.values())
        rel_to_tail_dist[r] = {
            t: round(c / total, 4)
            for t, c in sorted(tail_counts.items(), key=lambda x: -x[1])
        }
    return {
        "rel_head_to_ranked_tails": dict(rel_head_to_ranked_tails),
        "rel_to_tail_dist":         rel_to_tail_dist,
        "rel_to_tail_counts":       dict(rel_to_tail_counts),
        "rel_to_head_counts":       dict(rel_to_head_counts),
    }

def get_type_constraint_signal(head, relation, true_tail, predicted, constraints):
    rh_tails       = constraints["rel_head_to_ranked_tails"]
    tail_dist      = constraints["rel_to_tail_dist"]
    specific       = rh_tails.get((relation, head), {})
    general        = tail_dist.get(relation, {})
    type_fit_true  = specific.get(true_tail) or general.get(true_tail, 0.0)
    type_fit_pred  = specific.get(predicted) or general.get(predicted, 0.0)
    expected_tails = list(specific.keys())[:5] or list(general.keys())[:5]
    tail_counts    = constraints["rel_to_tail_counts"]
    true_rels = {r for r, tails in tail_counts.items() if true_tail in tails}
    pred_rels = {r for r, tails in tail_counts.items() if predicted  in tails}
    return {
        "type_fit_true":  type_fit_true,
        "type_fit_pred":  type_fit_pred,
        "type_gap":       round(type_fit_true - type_fit_pred, 4),
        "expected_tails": expected_tails,
        "only_true_has":  sorted(true_rels - pred_rels),
        "only_pred_has":  sorted(pred_rels - true_rels),
        "shared_rels":    sorted(true_rels & pred_rels),
    }


In [35]:
def failure_summary(head, relation, true_tail, predicted_tail, constraints):
    """Uses score_triple() with raw embeddings — no model object needed."""
    s  = label_to_entity_id[head]
    p  = label_to_relation_id[relation]
    o  = label_to_entity_id[true_tail]
    p2 = label_to_entity_id[predicted_tail]

    score_true = score_triple(s, p, o)
    score_pred = score_triple(s, p, p2)

    sig = get_type_constraint_signal(head, relation, true_tail, predicted_tail, constraints)
    summary = (
        f"Model predicted '{predicted_tail}' (score={score_pred:.3f}) "
        f"over '{true_tail}' (score={score_true:.3f}). "
        f"Type fit: '{true_tail}'={sig['type_fit_true']:.3f} vs "
        f"'{predicted_tail}'={sig['type_fit_pred']:.3f} for '{relation}'. "
        f"Expected tails: {', '.join(sig['expected_tails'][:3])}. "
        f"Only '{true_tail}' in: {', '.join(sig['only_true_has'][:5]) or 'none'}. "
        f"Only '{predicted_tail}' in: {', '.join(sig['only_pred_has'][:5]) or 'none'}."
    )
    return summary, {
        "score_true":     score_true,
        "score_pred":     score_pred,
        "shared":         set(sig["shared_rels"]),
        "only_true":      set(sig["only_true_has"]),
        "only_pred":      set(sig["only_pred_has"]),
        "type_fit_true":  sig["type_fit_true"],
        "type_fit_pred":  sig["type_fit_pred"],
        "type_gap":       sig["type_gap"],
        "expected_tails": sig["expected_tails"],
    }


In [36]:
@torch.no_grad()
def batch_rank_triples(tuples_batch: list, hard_threshold: int) -> list:
    valid = []
    for i, (h, r, t) in enumerate(tuples_batch):
        s = label_to_entity_id.get(h)
        p = label_to_relation_id.get(r)
        o = label_to_entity_id.get(t)
        if None in (s, p, o):
            valid.append((i, None))
        else:
            valid.append((i, (s, p, o, h, t)))

    good = [(i, d) for i, d in valid if d is not None]
    out  = [None] * len(tuples_batch)
    if not good:
        return out

    s_ids = torch.tensor([d[0] for _, d in good], device=DEVICE)
    p_ids = torch.tensor([d[1] for _, d in good], device=DEVICE)

    s_re = ENT_RE[s_ids]
    s_im = ENT_IM[s_ids]
    p_re = relation_emb_gpu[p_ids, :DIM]
    p_im = relation_emb_gpu[p_ids, DIM:]

    scores = (s_re * p_re - s_im * p_im) @ ENT_RE.T \
           + (s_re * p_im + s_im * p_re) @ ENT_IM.T

    for row_pos, (orig_idx, d) in enumerate(good):
        s, p, o, h_label, t_label = d
        row = scores[row_pos].clone()
        row[s] = float("-inf")

        ranked_ids  = torch.argsort(row, descending=True).cpu()
        scores_cpu  = row.cpu().numpy()
        ranked_list = ranked_ids.tolist()
        true_rank   = ranked_list.index(o) + 1

        out[orig_idx] = {
            "predicted":       id_to_entity_label[ranked_list[0]],
            "predicted_score": float(scores_cpu[ranked_list[0]]),
            "true_tail":       t_label,
            "true_score":      float(scores_cpu[o]),
            "true_rank":       true_rank,
            "full_ranking":    [(id_to_entity_label[i], round(float(scores_cpu[i]), 3))
                                for i in ranked_list[:200]],
            "hard_failure":    bool(true_rank > hard_threshold),
        }

    return out

# The training Loop that Helps in subsetting samples that fail before creating held_out set vs val_set 

In [37]:
import os, json, time
import torch
import torch.nn.functional as F
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

entity_emb_gpu   = entity_emb.to(DEVICE).float()
relation_emb_gpu = relation_emb.to(DEVICE).float()
N_ENT = entity_emb_gpu.shape[0]
DIM   = entity_emb_gpu.shape[1] // 2

ENT_RE = entity_emb_gpu[:, :DIM].contiguous()
ENT_IM = entity_emb_gpu[:, DIM:].contiguous()

_mag      = torch.sqrt(ENT_RE**2 + ENT_IM**2)
EMBS_NORM = F.normalize(_mag, dim=-1).contiguous()
EMBS_CACHE = _mag.cpu()

print("Building entity-relation cache …")
ent_to_rels: dict[str, set] = {}
for _, row in df_train.iterrows():
    h, r, t = row["head"], row["relation"], row["tail"]
    ent_to_rels.setdefault(h, set()).add(r)
    ent_to_rels.setdefault(t, set()).add(r)
print(f"  cached {len(ent_to_rels)} entities")

# Pre-cache head → relation → list of tails from training
ent_rel_to_tails: dict[tuple, list] = {}
for _, row in df_train.iterrows():
    key = (row["head"], row["relation"])
    ent_rel_to_tails.setdefault(key, []).append(row["tail"])
print(f"  cached {len(ent_rel_to_tails)} (entity, relation) pairs")

constraints = build_type_constraints(df_train)


def is_hard_failure(ranking, head, relation, constraints, hop_type="multi"):
    if ranking["true_rank"] == 1: return False
    if hop_type == "none":        return False
    return ranking["true_rank"] <= 12


def reclassify_hard(records, constraints):
    reclassified = 0
    for r in records:
        fake_ranking = {
            "true_rank":       r["true_rank"],
            "true_tail":       r["tail"],
            "predicted":       r["predicted"],
            "true_score":      r["score_true"],
            "predicted_score": r["score_predicted"],
        }
        new_hard = is_hard_failure(
            fake_ranking,
            r["head"],
            r["relation"],
            constraints,
            hop_type=r.get("hop_type", "multi")
        )
        if new_hard != r["hard_failure"]:
            r["hard_failure"] = new_hard
            reclassified += 1
    print(f"Reclassified {reclassified} records")
    return records


@torch.no_grad()
def batch_similarity_summary(entity_labels: list, k: int = 5) -> list:
    ids  = torch.tensor([label_to_entity_id[e] for e in entity_labels], device=DEVICE)
    vecs = EMBS_NORM[ids]
    sim_matrix = vecs @ EMBS_NORM.T
    topk_vals, topk_ids = torch.topk(sim_matrix, k + 1, dim=-1)
    topk_ids_cpu  = topk_ids.cpu().numpy()
    topk_vals_cpu = topk_vals.cpu().numpy()
    results = []
    for b_idx, entity in enumerate(entity_labels):
        e_rels  = ent_to_rels.get(entity, set())
        entries = []
        for j in range(topk_ids_cpu.shape[1]):
            idx  = int(topk_ids_cpu[b_idx, j])
            name = id_to_entity_label[idx]
            if name == entity:
                continue
            score  = float(topk_vals_cpu[b_idx, j])
            n_rels = ent_to_rels.get(name, set())
            shared = len(e_rels & n_rels)
            total  = len(e_rels | n_rels)
            rel_sc = shared / total if total > 0 else 0.0
            entries.append((name, score, shared, rel_sc))
            if len(entries) == k:
                break
        parts   = [f"{n}(sim={s:.2f},shared={sh},rel={rs:.2f})" for n, s, sh, rs in entries]
        summary = f"{entity} most similar to: {', '.join(parts)}"
        results.append((summary, entries))
    return results


def _cpu_record(h, r, t, ranking):
    sg = focused_subgraph(
        [h, t, ranking["predicted"]],
        graph, hops=3, max_triples=150, query_relation=r,
    )
    hop_type, hops, _ = hop_classifier(h, t, graph, target_relation=r)
    fail_sum, fail_raw = failure_summary(h, r, t, ranking["predicted"], constraints)
    return sg, hop_type, int(hops), fail_sum, fail_raw


def preprocess_all_triples_gpu(df, split_name, batch_size=128, cpu_workers=4):
    records        = []
    n_entities     = len(label_to_entity_id)
    hard_threshold = max(10, int(n_entities * 0.15))
    rows           = [(r["head"], r["relation"], r["tail"]) for _, r in df.iterrows()]
    total          = len(rows)
    t0             = time.time()

    for batch_start in range(0, total, batch_size):
        batch = rows[batch_start: batch_start + batch_size]
        valid = [
            (h, r, t) for h, r, t in batch
            if h in label_to_entity_id
            and t in label_to_entity_id
            and r in label_to_relation_id
        ]
        if not valid:
            continue

        rankings = batch_rank_triples(valid, hard_threshold)

        heads = [h for h, r, t in valid]
        tails = [t for h, r, t in valid]
        sims  = batch_similarity_summary(heads + tails, k=5)
        sim_heads = sims[:len(heads)]
        sim_tails = sims[len(heads):]

        def _work(args):
            i, (h, r, t) = args
            if rankings[i] is None:
                return i, None
            try:
                return i, _cpu_record(h, r, t, rankings[i])
            except Exception as e:
                print(f"\n  [CPU ERR] {h} | {r} | {t}: {e}")
                return i, None

        cpu_out = {}
        with ThreadPoolExecutor(max_workers=cpu_workers) as ex:
            for i, res in ex.map(_work, enumerate(valid)):
                cpu_out[i] = res

        for i, (h, r, t) in enumerate(valid):
            rk  = rankings[i]
            cpu = cpu_out.get(i)
            if rk is None or cpu is None:
                continue
            sg, hop_type, hops, fail_sum, fail_raw = cpu

            records.append({
                "split":            split_name,
                "head":             h,
                "relation":         r,
                "tail":             t,
                "true_rank":        int(rk["true_rank"]),
                "predicted":        rk["predicted"],
                "score_true":       float(rk["true_score"]),
                "score_predicted":  float(rk["predicted_score"]),
                "top5":             rk["full_ranking"][:15],
                "hop_type":         hop_type,
                "hop_count":        hops,
                "sim_head":         sim_heads[i][0],
                "sim_tail":         sim_tails[i][0],
                "fail_summary":     fail_sum,
                "subgraph":         sg,
                "shared_relations": list(fail_raw["shared"]),
                "only_tail_has":    list(fail_raw["only_true"]),
                "only_pred_has":    list(fail_raw["only_pred"]),
                "hard_failure":     is_hard_failure(
                                        rk, h, r, constraints,
                                        hop_type=hop_type),
                "known_tails":      ent_rel_to_tails.get((h, r), []),
            })

        done    = min(batch_start + batch_size, total)
        elapsed = time.time() - t0
        rate    = done / elapsed if elapsed > 0 else 0
        eta     = (total - done) / rate if rate > 0 else 0
        print(f"  {split_name}: {done}/{total}  "
              f"records={len(records)}  "
              f"{rate:.1f} rows/s  ETA {eta/60:.1f}min", end="\r")

    print()
    return records


# ── DELETE old files so preprocessing always runs fresh ──────────────────────
PREPROCESSED_TRAIN = "/kaggle/working/CODEX_S_preprocessed_train.json"
PREPROCESSED_TEST  = "/kaggle/working/CODEX_S_preprocessed_test.json"

for f in [PREPROCESSED_TRAIN, PREPROCESSED_TEST]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")

# ── Always preprocess fresh ───────────────────────────────────────────────────
t0 = time.time()
print("Preprocessing train …")
train_records = preprocess_all_triples_gpu(df_train, "train", batch_size=128)
print("Preprocessing test …")
test_records  = preprocess_all_triples_gpu(df_test,  "test",  batch_size=128)
with open(PREPROCESSED_TRAIN, "w") as f: json.dump(train_records, f, indent=2)
with open(PREPROCESSED_TEST,  "w") as f: json.dump(test_records,  f, indent=2)
print(f"\nDone in {(time.time()-t0)/60:.1f} min")

if EMBS_CACHE is None:
    EMBS_CACHE = _mag.cpu()

# ── Verify top5 now has 15 entries ────────────────────────────────────────────
sample = test_records[0]
print(f"\nVerification — top5 length: {len(sample['top5'])}  ← must be 15")
print(f"true_rank of sample: {sample['true_rank']}")
print(f"Correct answer visible: {sample['true_rank'] <= len(sample['top5'])}")
print(f"known_tails sample: {sample['known_tails'][:5]}")

Device: cuda  |  GPU: Tesla T4
Building entity-relation cache …
  cached 2034 entities
  cached 10465 (entity, relation) pairs
Preprocessing train …
  train: 32888/32888  records=32888  390.9 rows/s  ETA 0.0min
Preprocessing test …
  test: 1828/1828  records=1828  395.8 rows/s  ETA 0.0min

Done in 1.8 min

Verification — top5 length: 15  ← must be 15
true_rank of sample: 1
Correct answer visible: True
known_tails sample: []


In [38]:
# ── Diagnostic — understand what happened ─────────────────────────────────────
from collections import Counter

total_test     = len(test_records)
hard_all       = [r for r in test_records if r["hard_failure"]]
none_hop_all   = [r for r in test_records if r.get("hop_type") == "none"]
hard_none_hop  = [r for r in test_records if r["hard_failure"] and r.get("hop_type") == "none"]

print(f"Total test:          {total_test}")
print(f"hop_type=none:       {len(none_hop_all)}")
print(f"hard_failure=True:   {len(hard_all)}")
print(f"hard + hop=none:     {len(hard_none_hop)}  ← should be 0")

# Break down by hop type
hop_counts = Counter(r.get("hop_type") for r in hard_all)
print(f"\nHard failures by hop_type:")
for h, c in hop_counts.most_common():
    print(f"  {h}: {c}")

# Break down by rank
rank_counts = Counter(r["true_rank"] for r in hard_all)
print(f"\nHard failures by rank:")
for rank in sorted(rank_counts):
    print(f"  rank {rank}: {rank_counts[rank]}")

# Break down by score gap
gaps = [abs(r["score_predicted"] - r["score_true"]) for r in test_records
        if r.get("hop_type") != "none" and r["true_rank"] > 1 and r["true_rank"] <= 10]
print(f"\nNon-none hop, rank 2-10, score gaps:")
print(f"  < 0.5:  {sum(1 for g in gaps if g < 0.5)}")
print(f"  < 1.0:  {sum(1 for g in gaps if g < 1.0)}")
print(f"  < 2.0:  {sum(1 for g in gaps if g < 2.0)}")
print(f"  any:    {len(gaps)}")

Total test:          1828
hop_type=none:       1029
hard_failure=True:   196
hard + hop=none:     0  ← should be 0

Hard failures by hop_type:
  multi: 196

Hard failures by rank:
  rank 2: 36
  rank 3: 21
  rank 4: 20
  rank 5: 9
  rank 6: 14
  rank 7: 23
  rank 8: 16
  rank 9: 18
  rank 10: 12
  rank 11: 12
  rank 12: 15

Non-none hop, rank 2-10, score gaps:
  < 0.5:  4
  < 1.0:  13
  < 2.0:  33
  any:    169


In [39]:
# Exact count at each threshold across ALL non-none hop rank 2-10
gaps_detail = [(r["true_rank"], 
                abs(r["score_predicted"] - r["score_true"]),
                constraints["rel_to_tail_dist"].get(r["relation"], {}).get(r["tail"], 0.0))
               for r in test_records
               if r.get("hop_type") != "none" 
               and r["true_rank"] > 1 
               and r["true_rank"] <= 10]

print(f"Total rank 2-10, hop≠none: {len(gaps_detail)}")
print(f"\nBy gap threshold:")
for thresh in [0.5, 1.0, 2.0, 3.0, 5.0, 10.0, 999]:
    count = sum(1 for _, g, _ in gaps_detail if g < thresh)
    print(f"  gap < {thresh:5.1f}: {count}")

print(f"\nBy type_fit threshold:")
for thresh in [0.01, 0.02, 0.05, 0.10, 0.20]:
    count = sum(1 for _, _, tf in gaps_detail if tf < thresh)
    print(f"  type_fit < {thresh}: {count}")

print(f"\nBy rank only:")
for max_rank in range(2, 11):
    count = sum(1 for rank, _, _ in gaps_detail if rank <= max_rank)
    print(f"  rank <= {max_rank}: {count}")

Total rank 2-10, hop≠none: 169

By gap threshold:
  gap <   0.5: 4
  gap <   1.0: 13
  gap <   2.0: 33
  gap <   3.0: 69
  gap <   5.0: 147
  gap <  10.0: 169
  gap < 999.0: 169

By type_fit threshold:
  type_fit < 0.01: 17
  type_fit < 0.02: 40
  type_fit < 0.05: 88
  type_fit < 0.1: 130
  type_fit < 0.2: 149

By rank only:
  rank <= 2: 36
  rank <= 3: 57
  rank <= 4: 77
  rank <= 5: 86
  rank <= 6: 100
  rank <= 7: 123
  rank <= 8: 139
  rank <= 9: 157
  rank <= 10: 169


In [40]:
VAL_FILE        = "/kaggle/working/CODEX_S_val.json"
HELD_OUT_FILE   = "/kaggle/working/CODEX_S_held_out.json"
CHECKPOINT_FILE = "/kaggle/working/val_hard_checkpoint.json"
RESULTS_FILE    = "/kaggle/working/val_hard_results.json"

# ── Redefine correct version here to overwrite the bad one ───────────────────
def reclassify_hard(records, constraints):
    reclassified = 0
    for r in records:
        fake_ranking = {
            "true_rank":       r["true_rank"],
            "true_tail":       r["tail"],
            "predicted":       r["predicted"],
            "true_score":      r["score_true"],
            "predicted_score": r["score_predicted"],
        }
        new_hard = is_hard_failure(
            fake_ranking,
            r["head"],
            r["relation"],
            constraints,
            hop_type=r.get("hop_type", "multi")  # ← this is the critical line
        )
        if new_hard != r["hard_failure"]:
            r["hard_failure"] = new_hard
            reclassified += 1
    print(f"Reclassified {reclassified} records")
    return records

# ── Reclassify again with the now-correct function ────────────────────────────
constraints   = build_type_constraints(df_train)
train_records = reclassify_hard(train_records, constraints)
test_records  = reclassify_hard(test_records,  constraints)

# ── Delete old files ──────────────────────────────────────────────────────────
for f in [VAL_FILE, HELD_OUT_FILE, CHECKPOINT_FILE, RESULTS_FILE]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")

# ── Rebuild split ─────────────────────────────────────────────────────────────
val_records, held_out_records = train_test_split(
    test_records, test_size=0.5, random_state=42
)
with open(VAL_FILE,      "w") as f: json.dump(val_records,      f, indent=2)
with open(HELD_OUT_FILE, "w") as f: json.dump(held_out_records, f, indent=2)

val_hard = [r for r in val_records if r["hard_failure"]]

# ── Sanity check ──────────────────────────────────────────────────────────────
none_in_hard    = [r for r in val_hard if r.get("hop_type") == "none"]
rank_violations = [r for r in val_hard if r["true_rank"] > 12]
print(f"hop_type=none in val_hard: {len(none_in_hard)}  ← must be 0")
print(f"rank > 10 in val_hard:     {len(rank_violations)}  ← must be 0")
print(f"val_hard size:             {len(val_hard)}  ← expect ~100-110")
print(f"Type constraints:          {len(constraints['rel_to_tail_dist'])} relations")

Reclassified 0 records
Reclassified 0 records
hop_type=none in val_hard: 0  ← must be 0
rank > 10 in val_hard:     0  ← must be 0
val_hard size:             101  ← expect ~100-110
Type constraints:          42 relations


In [41]:
def count_hard(records, constraints):
    hard, breakdown = [], {"top10_gap": 0, "top10_type": 0, "both": 0}
    for r in records:
        true_rank  = r["true_rank"]
        score_gap  = abs(r["score_predicted"] - r["score_true"])
        type_fit   = constraints["rel_to_tail_dist"].get(
                         r["relation"], {}).get(r["tail"], 0.0)

        if true_rank == 1:
            continue

        in_top10     = true_rank <= 10
        gap_small    = score_gap < 0.5
        low_type_fit = type_fit < 0.02

        if in_top10 and (gap_small or low_type_fit):
            hard.append(r)
            if gap_small and low_type_fit: breakdown["both"] += 1
            elif gap_small:                breakdown["top10_gap"] += 1
            else:                          breakdown["top10_type"] += 1

    return hard, breakdown

hard_check, bd = count_hard(test_records, constraints)
print(f"Hard failures:  {len(hard_check)}")
print(f"  gap < 0.5 only:      {bd['top10_gap']}")
print(f"  type_fit < 0.02 only:{bd['top10_type']}")
print(f"  both conditions:     {bd['both']}")

# Distribution of ranks in hard set
from collections import Counter
rank_dist = Counter(r["true_rank"] for r in hard_check)
for rank in sorted(rank_dist):
    print(f"  rank {rank}: {rank_dist[rank]} cases")

Hard failures:  213
  gap < 0.5 only:      13
  type_fit < 0.02 only:199
  both conditions:     1
  rank 2: 20 cases
  rank 3: 17 cases
  rank 4: 18 cases
  rank 5: 17 cases
  rank 6: 30 cases
  rank 7: 21 cases
  rank 8: 19 cases
  rank 9: 35 cases
  rank 10: 36 cases


# For Faster retrival of episodic triples in the trianing Phase we use FIASS vector database which increases the rate of retrival, mainly to help the model as its context engineering hence these results get appended during training and hence helps in faster system lower training time 

In [42]:
import csv
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings   import HuggingFaceEmbeddings

EMBED         = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
EPISODIC_PATH = "/kaggle/working/episodic.faiss"
SEMANTIC_PATH = "/kaggle/working/semantic.faiss"
EPISODIC_TSV  = "/kaggle/working/episodic_memory.tsv"

def _load_or_create_faiss(path):
    if os.path.exists(path):
        return FAISS.load_local(path, EMBED, allow_dangerous_deserialization=True)
    store = FAISS.from_texts(["init"], EMBED)
    store.save_local(path)
    return store

episodic_vs = _load_or_create_faiss(EPISODIC_PATH)
semantic_vs = _load_or_create_faiss(SEMANTIC_PATH)

def query_episodic(head, relation):
    docs = episodic_vs.similarity_search(f"{head} {relation}", k=2)
    return "\n".join(d.page_content for d in docs if "init" not in d.page_content)

def query_semantic(failure_type):
    docs = semantic_vs.similarity_search(failure_type, k=2)
    return "\n".join(d.page_content for d in docs if "init" not in d.page_content)

def write_episodic(head, relation, tail, agent, finding):
    text = f"{head} {relation} → {tail} | agent {agent} | {finding}"
    episodic_vs.add_texts([text])
    episodic_vs.save_local(EPISODIC_PATH)

def write_semantic(agent, failure_type, key_rels):
    text = f"agent {agent} wins on {failure_type} | key_rels: {', '.join(key_rels)}"
    semantic_vs.add_texts([text])
    semantic_vs.save_local(SEMANTIC_PATH)

def load_tsv_memory(path=EPISODIC_TSV):
    memory = defaultdict(list)
    if not os.path.exists(path):
        return memory
    with open(path) as f:
        for row in csv.DictReader(f, delimiter="\t"):
            memory[row["head"].strip()].append(
                (row["relation"].strip(), row["tail"].strip())
            )
    return memory

def get_memory_hint(head, memory, relation=None, k=5):
    triples = memory.get(head, [])
    if not triples:
        return ""
    if relation:
        same = [(r, t) for r, t in triples if r == relation]
        rest = [(r, t) for r, t in triples if r != relation]
        selected = same[:k] + rest[:max(0, k - len(same))]
    else:
        selected = triples[:k]
    seen, unique = set(), []
    for r, t in selected:
        if (r, t) not in seen:
            seen.add((r, t)); unique.append((r, t))
    return "\n".join(f"{head} --{r}--> {t}" for r, t in unique)

tsv_memory = load_tsv_memory()
print(f"TSV memory: {len(tsv_memory)} entities  "
      f"{sum(len(v) for v in tsv_memory.values())} triples")

/tmp/ipykernel_55/1781542869.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  EMBED         = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TSV memory: 0 entities  0 triples


# Qwen Instruct the backbone LLM for all the 3 agents , for faster downloads use HF token

In [43]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, pipeline as hf_pipeline,
)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {MODEL_ID} in 4-bit …")

_tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
_mdl = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    ),
    device_map="auto",
    trust_remote_code=True,
)
# REMOVE max_length — it conflicts with max_new_tokens
_pipe = hf_pipeline(
    "text-generation",
    model=_mdl,
    tokenizer=_tok,
    do_sample=True,
    temperature=0.3,
    return_full_text=False,
    max_new_tokens=512,     # ← only set this, not max_length
)
print(f"LLM ready on {_mdl.device}")

def call_llm(system: str, user: str) -> str:
    messages = [{"role": "system", "content": system},
                {"role": "user",   "content": user}]
    for attempt in range(2):
        try:
            return _pipe(messages)[0]["generated_text"].strip()
        except Exception as e:
            if attempt == 0:
                print(f"  [LLM] retry: {e}"); time.sleep(5)
            else:
                raise

def parse_json(raw: str, who: str) -> dict:
    text = raw.strip()
    if text.startswith("```"):
        text = "\n".join(text.split("\n")[1:-1])
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"  [{who}] JSON parse failed: {e}\n  raw: {raw[:200]}")
        return {"error": str(e), "raw": raw}

Loading Qwen/Qwen2.5-7B-Instruct in 4-bit …


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM ready on cuda:0


# Agen A and Agent B prompts and Aggregator prompts (Context + reAct based prompting)

In [44]:
_A_SYS = """You are Agent A, a knowledge graph reasoning agent.

Your task: predict the correct tail entity for the given triple.
The model got it wrong — use type constraints and training evidence to fix it.

CRITICAL REASONING RULES:
1. Check TRAINING EVIDENCE first — these tails are already known for this entity+relation
2. The correct answer is likely NOT in the training evidence (that is why the model missed it)
3. From the CANDIDATES list, eliminate any that appear in training evidence
4. Among remaining candidates, pick the one with best type fit for this relation
5. If all candidates are in training evidence, pick the one with highest type fit

IMPORTANT FORMAT RULE for shared_relations:
- CORRECT:   "shared_relations": ["field of work"]
- WRONG:     "shared_relations": ["occupation --> politician"]
Only plain relation names. Never paths. Never arrows.

Output ONLY valid JSON:
{
  "prediction": "<entity name>",
  "confidence": <0.0-1.0>,
  "shared_relations": ["<subgraph relations supporting prediction>"],
  "failure_diagnosis": "<why model got this wrong, cite training evidence>",
  "evidence_type": "type_constraint | subgraph | training_evidence | mixed"
}"""


_B_SYS = """You are Agent B, a knowledge graph path reasoning agent.

Your task: predict the correct tail entity for the given triple.
The model got it wrong — use subgraph paths and training evidence to fix it.

CRITICAL REASONING RULES:
1. Check TRAINING EVIDENCE first — these tails are already known for this entity+relation
2. The correct answer is likely NOT in the training evidence
3. From the CANDIDATES list, find one that is reachable via subgraph path
4. Prefer candidates NOT in training evidence that have valid subgraph paths
5. If no path exists, use type constraints to pick from non-training candidates

IMPORTANT FORMAT RULE:
Only cite relations that appear in the subgraph.
Never cite paths as relation names.

Output ONLY valid JSON:
{
  "prediction": "<entity name>",
  "confidence": <0.0-1.0>,
  "key_relations": ["<subgraph relations supporting prediction>"],
  "path_found": "<path from subgraph or null>",
  "path_relation_matches_query": <true|false>,
  "reasoning": "<cite training evidence + subgraph evidence>",
  "failure_diagnosis": "<why model got this wrong>",
  "evidence_type": "subgraph_path | type_constraint | training_evidence | mixed"
}"""

In [45]:
_USER_TMPL = """
{context}

<episodic_memory>
{episodic_hint}
</episodic_memory>

<reasoning>
STEP 1 — CHECK TRAINING EVIDENCE
  What tails are already known for this (head, relation) pair?
  The correct answer is probably NOT one of those known tails.

STEP 2 — ELIMINATE FROM CANDIDATES
  Remove any candidate that appears in the training evidence.
  The remaining candidates are your real options.

STEP 3 — GOLD SIGNAL (if ◆ exists)
  Which remaining candidate participates in ◆ relations?
  That is most likely correct.

STEP 4 — SUBGRAPH PATH
  Is there a path from head to any remaining candidate?
  Prefer candidates reachable via the query relation.

STEP 5 — TYPE CONSTRAINTS
  Among remaining candidates, which has highest type fit?

STEP 6 — COMMIT
  training_evidence elimination + gold signal → 0.85-0.95
  training_evidence elimination + path found  → 0.80-0.90
  training_evidence elimination + type fit    → 0.65-0.80
  type fit only                               → 0.50-0.65
</reasoning>

Respond with valid JSON only:
"""

In [46]:

def trim_subgraph(subgraph, head, true_tail, predicted, only_tail_has, max_triples=12):
    oth_set = set(only_tail_has)
    tier1, tier2, tier3, tier4 = [], [], [], []
    for triple in subgraph:
        if isinstance(triple, (list, tuple)) and len(triple) == 3:
            h, r, t = triple
        elif isinstance(triple, dict):
            h = triple.get("head",""); r = triple.get("relation",""); t = triple.get("tail","")
        else:
            continue
        if r in oth_set:                                          tier1.append(triple)
        elif t in (true_tail, predicted) or h in (true_tail, predicted): tier2.append(triple)
        elif h == head or t == head:                              tier3.append(triple)
        else:                                                     tier4.append(triple)
    return (tier1 + tier2 + tier3 + tier4)[:max_triples]

def build_subgraph_str(record, max_triples=12):
    true_tail = record.get("true_tail") or record.get("tail", "")
    trimmed   = trim_subgraph(
        record.get("subgraph", []), record["head"], true_tail,
        record.get("predicted",""), record.get("only_tail_has",[]), max_triples,
    )
    if not trimmed:
        return "  (no subgraph available)"
    only_set = set(record.get("only_tail_has", []))
    lines = []
    for triple in trimmed:
        if isinstance(triple, (list, tuple)) and len(triple) == 3:
            h, r, t = triple
        elif isinstance(triple, dict):
            h = triple.get("head","?"); r = triple.get("relation","?"); t = triple.get("tail","?")
        else:
            continue
        lines.append(f"  {h} --{r}--> {t}" + (" ◆" if r in only_set else ""))
    return "\n".join(lines)

# Builds Agent context 


In [47]:
def build_agent_context(record, tsv_memory=None):
    true_tail = record.get("true_tail") or record.get("tail", "")
    predicted = record.get("predicted", "unknown")
    only_tail = record.get("only_tail_has", [])
    only_pred = record.get("only_pred_has", [])
    subgraph  = record.get("subgraph", [])
    true_rank = record.get("true_rank", 0)

    # ── Subgraph inventory ────────────────────────────────────────────────────
    subgraph_relations = set()
    subgraph_entities  = set()
    for triple in subgraph:
        if isinstance(triple, (list, tuple)) and len(triple) == 3:
            h, r, t = triple
        elif isinstance(triple, dict):
            h = triple.get("head","")
            r = triple.get("relation","")
            t = triple.get("tail","")
        else:
            continue
        subgraph_relations.add(r)
        subgraph_entities.update([h, t])

    # ── Candidates with training evidence pre-filtered ────────────────────────
    top_n       = record.get("top5", [])[:15]
    known_tails = set(record.get("known_tails", []))

    eliminated = [(i+1, e, s) for i, (e, s) in enumerate(top_n)
                  if e in known_tails]
    remaining  = [(i+1, e, s) for i, (e, s) in enumerate(top_n)
                  if e not in known_tails]

    in_top_n   = true_rank <= len(top_n)

    if remaining:
        candidates_str = (
            "VALID CANDIDATES (training evidence removed):\n"
            + "\n".join(f"  rank {rank}: {e}  (score={s:.3f})"
                        for rank, e, s in remaining)
        )
    else:
        candidates_str = (
            "  (all top candidates are in training evidence)\n"
            "  Use type constraints to pick the best entity for this relation."
        )

    if eliminated:
        eliminated_str = (
            "ELIMINATED — already known in training, do NOT predict these:\n"
            + "\n".join(f"  ✗ rank {rank}: {e}" for rank, e, s in eliminated)
            + "\n"
        )
    else:
        eliminated_str = ""

    if in_top_n:
        rank_hint = f"The correct answer IS in the valid candidates above (rank {true_rank})."
    else:
        rank_hint = (
            f"The correct answer is at rank {true_rank} — NOT in the list above.\n"
            f"Use subgraph and type constraints to reason what it should be."
        )

    # ── Gold signal block ─────────────────────────────────────────────────────
    gold_in_subgraph     = [r for r in only_tail if r in subgraph_relations]
    gold_not_in_subgraph = [r for r in only_tail if r not in subgraph_relations]

    if gold_in_subgraph:
        gold_block = (
            "★ GOLD SIGNAL (in subgraph ◆):\n"
            + "\n".join(f"  ◆ {r}" for r in gold_in_subgraph)
            + f"\nThese discriminate [{true_tail}] from [{predicted}].\n"
        )
    elif only_tail:
        gold_block = (
            "⚠ DISCRIMINATING RELATIONS exist but not in subgraph:\n"
            + "\n".join(f"  {r}" for r in only_tail)
            + "\nUse candidates + type constraints to decide.\n"
        )
    else:
        gold_block = (
            "⚠ NO DISCRIMINATING RELATIONS.\n"
            "Use valid candidates list + type constraints + subgraph paths.\n"
        )

    # ── Forbidden block ───────────────────────────────────────────────────────
    forbidden_block = ""
    if only_pred:
        forbidden_block = (
            "FORBIDDEN (connect to WRONG entity only):\n"
            + "\n".join(f"  ✗ {r}" for r in only_pred[:5]) + "\n"
        )

    # ── Memory block ──────────────────────────────────────────────────────────
    memory_block = ""
    if tsv_memory:
        hint = get_memory_hint(record["head"], tsv_memory)
        if hint:
            memory_block = f"\nVerified past triples for {record['head']}:\n{hint}\n"

    sim_head = record.get("sim_head", "")
    if sim_head:
        sim_head = ", ".join(sim_head.split(", ")[:3])

    return (
        f"━━━ TASK ━━━\n"
        f"Triple: ({record['head']}, {record['relation']}, ?)\n"
        f"Model predicted (WRONG): {predicted}\n"
        f"Rank of correct answer:  {true_rank}\n\n"

        f"━━━ CANDIDATES ━━━\n"
        f"{candidates_str}\n\n"
        f"{eliminated_str}"
        f"{rank_hint}\n\n"

        f"━━━ EVIDENCE ━━━\n"
        f"{gold_block}\n"
        f"{forbidden_block}"
        f"Subgraph relations: "
        f"{', '.join(sorted(subgraph_relations)[:12]) or 'none'}\n\n"

        f"━━━ SUBGRAPH ━━━\n"
        f"{build_subgraph_str(record)}\n\n"

        f"{memory_block}"
        f"━━━ SUPPORTING INFO ━━━\n"
        f"Embedding neighbours of head: {sim_head}\n"
        f"Hop type: {record.get('hop_type','multi')}\n"
        f"Failure detail: {record.get('fail_summary','')}"
    )

# Agent based Hallucination checks , helps the Model be grounded during Training Phase (Offline Phase)

In [48]:
def verify_type_fit(agent_out, head, relation, constraints):
    key_rels = agent_out.get("key_relations", [])
    if not key_rels:
        return {"verified": True, "hallucinated": [], "confirmed": []}
    all_rels     = set(constraints["rel_to_tail_counts"].keys())
    confirmed    = [r for r in key_rels if r     in all_rels]
    hallucinated = [r for r in key_rels if r not in all_rels]
    if hallucinated:
        print(f"  [verify B] hallucinated: {hallucinated}")
        agent_out["key_relations"] = confirmed
    else:
        print(f"  [verify B] all relations confirmed")
    return {"verified": len(hallucinated) == 0, "confirmed": confirmed, "hallucinated": hallucinated}

def verify_relations(claimed_rels, head, tail, df_ref, query_relation=None):
    if not claimed_rels:
        return {"verified": [], "hallucinated": [], "rate": 0.0}
    # exclude query relation — it's always valid by definition
    to_check     = [r for r in claimed_rels if r != query_relation]
    if not to_check:
        return {"verified": claimed_rels, "hallucinated": [], "rate": 0.0}
    actual       = set(df_ref[(df_ref["head"] == head) & (df_ref["tail"] == tail)]["relation"])
    verified     = [r for r in to_check if r in actual] + \
                   ([query_relation] if query_relation in claimed_rels else [])
    hallucinated = [r for r in to_check if r not in actual]
    return {"verified": verified, "hallucinated": hallucinated,
            "rate": len(hallucinated) / max(len(to_check), 1)}
    
def verify_against_subgraph(agent_out, record, who):
    subgraph = record.get("subgraph", [])
    subgraph_relations = set()
    for triple in subgraph:
        if isinstance(triple, (list, tuple)) and len(triple) == 3:
            subgraph_relations.add(triple[1])
        elif isinstance(triple, dict):
            subgraph_relations.add(triple.get("relation", ""))

    key = "shared_relations" if who == "A" else "key_relations"
    claimed = agent_out.get(key, [])

    grounded   = [r for r in claimed if r in subgraph_relations]
    ungrounded = [r for r in claimed if r not in subgraph_relations]

    if ungrounded:
        print(f"  [subgraph verify {who}] ✗ not in subgraph: {ungrounded}")
        agent_out[key] = grounded
        if not grounded:
            # Agent cited nothing real — zero out confidence
            agent_out["confidence"] = min(agent_out.get("confidence", 0.1), 0.15)
            print(f"  [subgraph verify {who}] confidence capped at 0.15")
    else:
        print(f"  [subgraph verify {who}] ✓ all relations grounded in subgraph")

    return {"grounded": grounded, "ungrounded": ungrounded}

# The main Component that gives the Grounded score this helps in making sure that the agent ouputs are grounded and helps in making a score to choose which agent found the best way to get that tail , not just hallucinate towards the tail, helps penalize lucky prediction which is the paramteric memory in the training phase 

In [49]:
def grounded_score(agent_out, record, who, df_ref=None, constraints=None):
    true_tail = record.get("true_tail") or record.get("tail", "")
    only_tail = set(record.get("only_tail_has", []))
    only_pred = set(record.get("only_pred_has", []))
    subgraph  = record.get("subgraph", [])
    path_verified, path_verification, rel_verification = None, {}, {}

    if who == "B" and constraints is not None:
        pv = verify_type_fit(agent_out, record["head"], record["relation"], constraints)
        path_verified     = pv["verified"]
        path_verification = pv
        if not pv["verified"]:
            agent_out["path_relation_matches_query"] = False
            agent_out["path_found"]    = "none"
            agent_out["key_relations"] = pv["confirmed"]

    if who == "A" and df_ref is not None:
        rv = verify_relations(
            agent_out.get("shared_relations", []), record["head"], true_tail, df_ref,query_relation=record.get("relation")
        )
        rel_verification = rv
        if rv["hallucinated"]:
            print(f"  [verify A] ✗ hallucinated: {rv['hallucinated']}")
            agent_out["shared_relations"] = rv["verified"]
        else:
            print(f"  [verify A] all relations confirmed")

    sg_verify = verify_against_subgraph(agent_out, record, who)

    subgraph_relations = set()
    subgraph_entities  = set()
    for triple in subgraph:
        if isinstance(triple, (list, tuple)) and len(triple) == 3:
            h, r, t = triple
        elif isinstance(triple, dict):
            h = triple.get("head","")
            r = triple.get("relation","")
            t = triple.get("tail","")
        else:
            continue
        subgraph_relations.add(r)
        subgraph_entities.update([h, t])

    agent_rels   = set(agent_out.get("shared_relations", []) if who == "A"
                       else agent_out.get("key_relations", []))
    overlap_tail = agent_rels & only_tail
    overlap_pred = agent_rels & only_pred

    prediction = agent_out.get("prediction", "") or ""
    confidence = agent_out.get("confidence",  0.0) or 0.0
    correct    = prediction.strip().lower() == true_tail.strip().lower()
    abstained  = prediction in [None, "null", "", "None"]

    if only_tail:
        coverage_score = len(overlap_tail) / len(only_tail)
    else:
        coverage_score = 0.0

    # ── COMPONENT 2: Subgraph grounding ──────────────────────────────────────
    if agent_rels and subgraph_relations:
        grounded_rels   = agent_rels & subgraph_relations
        grounding_score = len(grounded_rels) / len(agent_rels)
    elif abstained:
        grounding_score = 0.3
    else:
        grounding_score = 0.0

    if constraints and not abstained:
        tail_dist     = constraints["rel_to_tail_dist"].get(record["relation"], {})
        rh_tails      = constraints["rel_head_to_ranked_tails"]
        specific      = rh_tails.get((record["relation"], record["head"]), {})
        type_fit_pred = specific.get(prediction, tail_dist.get(prediction, 0.0))
        max_specific  = max(specific.values())  if specific  else 0.0
        max_general   = max(tail_dist.values()) if tail_dist else 0.0
        max_fit       = max(max_specific, max_general, 0.001)
        type_score    = min(type_fit_pred / max_fit, 1.0)   # hard clamp
    else:
        type_score = 0.3

    # Zero evidence penalty — agent predicted but cited nothing
    if not agent_rels and not abstained:
        type_score = type_score * 0.2
        print(f"  [score {who}] ⚠ zero relations cited — type score discounted")

    if who == "B":
        path_found   = agent_out.get("path_found", None)
        path_matches = agent_out.get("path_relation_matches_query", False)
        path_valid   = (path_found not in [None, "null", "none", ""]
                        and path_verified is not False)
        if path_valid and path_matches:   path_score = 1.0
        elif path_valid:                  path_score = 0.5
        else:                             path_score = 0.0
    else:
        path_score = 0.0

    evidence_strength = max(coverage_score, grounding_score, path_score)
    if abstained:
        confidence_score = 1.0 - confidence
    elif evidence_strength > 0.5:
        confidence_score = confidence
    else:
        confidence_score = max(0.0, 1.0 - confidence)
    if who == "B":
        quality_score = round(
            0.30 * coverage_score  +
            0.25 * grounding_score +
            0.20 * type_score      +
            0.15 * path_score      +
            0.10 * confidence_score, 3
        )
    else:
        quality_score = round(
            0.35 * coverage_score  +
            0.30 * grounding_score +
            0.25 * type_score      +
            0.10 * confidence_score, 3
        )

    contamination = (len(overlap_pred) / max(len(agent_rels), 1)
                     if agent_rels else 0.0)
    quality_score = round(quality_score * (1 - 0.3 * contamination), 3)

    subgraph_grounded = sorted(agent_rels & subgraph_relations)

    print(f"  [score {who}] "
          f"coverage={coverage_score:.2f}  "
          f"grounding={grounding_score:.2f}  "
          f"type={type_score:.2f}  "
          f"path={path_score:.2f}  "
          f"calib={confidence_score:.2f}  "
          f"→ quality={quality_score}  "
          f"correct={correct}")

    return {
        "agent":               who,
        "quality_score":       quality_score,
        "coverage_score":      round(coverage_score,   3),
        "grounding_score":     round(grounding_score,  3),
        "type_score":          round(type_score,       3),
        "path_score":          round(path_score,       3),
        "confidence_score":    round(confidence_score, 3),
        "contamination":       round(contamination,    3),
        "relation_score":      round(coverage_score,   3),
        "overlap_tail":        sorted(overlap_tail),
        "overlap_pred":        sorted(overlap_pred),
        "agent_relations":     sorted(agent_rels),
        "subgraph_grounded":   subgraph_grounded,   # ← new field
        "only_tail_has":       sorted(only_tail),
        "prediction_correct":  correct,
        "confidence":          confidence,
        "path_verified":       path_verified,
        "path_verification":   path_verification,
        "rel_verification":    rel_verification,
    }

# MAIN FUNCTION - CALLS THE AGENTS + AGGREGATOR ( AGG IS THE POST HOC ANALYSIS) agent that reasonns why agent A or Agent B with the reasonign it produces which can be checked in the json file in the /kaggle/working 

In [50]:
from concurrent.futures import ThreadPoolExecutor

AGENT_A_B_DELAY = 1
BETWEEN_RECORDS = 1

def agent_a(context, record):
    print(f"  [A] ({record['head']}, {record['relation']}, {record.get('tail','')})")
    hint = query_episodic(record["head"], record["relation"])
    raw  = call_llm(_A_SYS, _USER_TMPL.format(context=context, episodic_hint=hint or "none"))
    out  = parse_json(raw, "A")
    print(f"  [A] pred={out.get('prediction')}  conf={out.get('confidence')}")
    return out

def agent_b(context, record):
    print(f"  [B] ({record['head']}, {record['relation']}, {record.get('tail','')})")
    hint = query_episodic(record["head"], record["relation"])
    raw  = call_llm(_B_SYS, _USER_TMPL.format(context=context, episodic_hint=hint or "none"))
    out  = parse_json(raw, "B")
    print(f"  [B] pred={out.get('prediction')}  conf={out.get('confidence')}")
    return out

def run_parallel_staggered(record, tsv_memory=None):
    context = build_agent_context(record, tsv_memory=tsv_memory)
    results = {}
    def _run_a(): results["A"] = agent_a(context, record)
    def _run_b(): time.sleep(AGENT_A_B_DELAY); results["B"] = agent_b(context, record)
    with ThreadPoolExecutor(max_workers=2) as ex:
        fa = ex.submit(_run_a); fb = ex.submit(_run_b)
        fa.result(); fb.result()
    return results["A"], results["B"]

def _python_route(s_a, s_b, record):
    qa, qb = s_a["quality_score"], s_b["quality_score"]
    ca, cb = s_a["contamination"],  s_b["contamination"]

    # NEW — if BOTH agents have zero quality and no gold signal exists,
    # don't pretend either agent did well
    only_tail = record.get("only_tail_has", [])
    if qa == 0.0 and qb == 0.0 and len(only_tail) == 0:
        # No discriminating signal existed — flag this explicitly
        record["_no_signal"] = True
        return "A"  # arbitrary, both equally bad

    if qa > 0.5 and qb <= 0.5: return "A"
    if qb > 0.5 and qa <= 0.5: return "B"
    if qa > 0 and qb > 0:
        if ca < cb: return "A"
        if cb < ca: return "B"
    return "A" if record["hop_type"] == "single" else "B"

def get_deciding_signal(s_a, s_b, a_out, b_out, chosen, record):
    qa, qb = s_a["quality_score"], s_b["quality_score"]
    ca, cb = s_a["contamination"],  s_b["contamination"]
    a_rels = s_a.get("agent_relations", [])
    b_rels = s_b.get("agent_relations", [])
    b_path = b_out.get("path_found", "none")
    if chosen == "A":
        if qa > 0.5 and qb <= 0.5:
            return f"A quality ({qa:.2f}) > threshold, B quality ({qb:.2f}) did not — A cited {a_rels}"
        if ca < cb:
            return f"A contam ({ca:.2f}) < B ({cb:.2f})"
        return f"Equal quality, single-hop favours A — cited {a_rels}"
    else:
        if qb > 0.5 and qa <= 0.5:
            return f"B quality ({qb:.2f}) > threshold, A quality ({qa:.2f}) did not"
        if cb < ca:
            return f"B contam ({cb:.2f}) < A ({ca:.2f}) — B path {b_path}"
        path_str = f"path {b_path} found" if b_path not in ["none","null",None,""] else "no path"
        return f"Equal quality, multi-hop — B {path_str}, cited {len(b_rels)} rels vs A {len(a_rels)}"

_AGG_SYS = """You are documenting a Knowledge Graph routing decision.
Output ONLY valid JSON:
{
  "reason": "<one specific, informative sentence>",
  "failure_type": "similarity_confusion | structural_gap | both_failed | resolved"
}"""

_AGG_USER = """Triple: ({head}, {relation}, ?)
Hop type: {hop_type}  |  Chosen agent: {chosen}
Deciding signal: {deciding_signal}
Agent A — prediction: {a_pred}  confidence: {a_conf}  shared_relations: {a_relations}  failure_diagnosis: {a_diagnosis}
Agent B — prediction: {b_pred}  confidence: {b_conf}  key_relations: {b_relations}  path_found: {b_path}  path_relation_matches_query: {b_rel_match}  reasoning: {b_reasoning}  failure_diagnosis: {b_diagnosis}
Write ONE sentence explaining why Agent {chosen} was trusted."""

def aggregate(a_out, b_out, s_a, s_b, record):
    chosen       = _python_route(s_a, s_b, record)
    chosen_out   = a_out if chosen == "A" else b_out
    chosen_score = s_a   if chosen == "A" else s_b

    # ── Short-circuit: no signal → don't let LLM hallucinate "resolved" ──────
    no_signal    = record.get("_no_signal", False)
    both_zero    = s_a["quality_score"] == 0.0 and s_b["quality_score"] == 0.0
    b_abstained  = (b_out.get("prediction") in [None, "null", ""] 
                    and b_out.get("confidence", 1.0) <= 0.10)

    if no_signal or (both_zero and b_abstained):
        print(f"  [Agg] SHORT-CIRCUIT — no signal, both agents have no evidence")
        return {
            "final_answer":       None,
            "chosen_agent":       "none",
            "confidence":         0.05,
            "reason":             "No discriminating signal in subgraph. Both agents had zero quality score.",
            "selected_relations": [],
            "failure_type":       "both_failed",
        }

    # ── Normal path ───────────────────────────────────────────────────────────
    final_answer    = chosen_out.get("prediction", "")
    deciding_signal = get_deciding_signal(s_a, s_b, a_out, b_out, chosen, record)

    print(f"  [Route] agent {chosen}  quality={chosen_score['quality_score']}  "
          f"contam={chosen_score['contamination']}")
    print(f"  [Route] {deciding_signal}")
    print("  [Agg] LLM labelling …")

    user = _AGG_USER.format(
        head=record["head"], relation=record["relation"],
        hop_type=record["hop_type"], chosen=chosen,
        deciding_signal=deciding_signal,
        a_pred=a_out.get("prediction","unknown"), a_conf=a_out.get("confidence",0.0),
        a_relations=a_out.get("shared_relations",[]),
        a_diagnosis=a_out.get("failure_diagnosis","none"),
        b_pred=b_out.get("prediction","unknown"), b_conf=b_out.get("confidence",0.0),
        b_relations=b_out.get("key_relations",[]),
        b_path=b_out.get("path_found","none"),
        b_rel_match=b_out.get("path_relation_matches_query",False),
        b_reasoning=b_out.get("reasoning","none"),
        b_diagnosis=b_out.get("failure_diagnosis","none"),
    )
    llm_out = parse_json(call_llm(_AGG_SYS, user), "Agg")

    # ── Extra guard: LLM said "resolved" but prediction is empty/null ─────────
    failure_type = llm_out.get("failure_type", "both_failed")
    if failure_type == "resolved" and not final_answer:
        failure_type = "both_failed"

    out = {
        "final_answer":       final_answer,
        "chosen_agent":       chosen,
        "confidence":         chosen_out.get("confidence", 0.0),
        "reason":             llm_out.get("reason", ""),
        "selected_relations": (chosen_score.get("overlap_tail", []) or
                       chosen_score.get("subgraph_grounded", [])),
        "failure_type":       failure_type,
    }
    print(f"  [Agg] final={out['final_answer']}  agent={out['chosen_agent']}  "
          f"type={out['failure_type']}")
    return out

def extract_reasoning_patterns(record, agg, constraints=None):
    patterns  = []
    head      = record["head"]
    true_tail = record.get("true_tail") or record.get("tail", "")
    only_tail = set(record.get("only_tail_has", []))
    selected  = set(agg.get("selected_relations", []))
    grounded  = selected & only_tail
    if constraints is not None:
        all_rels = set(constraints["rel_to_tail_counts"].keys())
        for rel_r in grounded:
            if rel_r in all_rels:
                patterns.append((head, rel_r, true_tail))
    return patterns

In [51]:
def writeback(agg, record, constraints=None):
    true_tail = record.get("true_tail") or record.get("tail", "")

    if agg.get("failure_type") != "resolved":
        return
    if agg.get("final_answer","").strip().lower() != true_tail.strip().lower():
        return

    # ── Try gold discriminative patterns first ────────────────────────────────
    patterns = extract_reasoning_patterns(record, agg, constraints=constraints)

    # ── Fall back to any subgraph-grounded relations ──────────────────────────
    if not patterns:
        selected = agg.get("selected_relations", [])
        if selected and constraints is not None:
            all_rels = set(constraints["rel_to_tail_counts"].keys())
            for rel_r in selected:
                if rel_r in all_rels:
                    patterns.append((record["head"], rel_r, true_tail))

    if not patterns:
        print("  [TSV] skipped — no grounded relations to write")
        return

    new_file = not os.path.exists(EPISODIC_TSV)
    with open(EPISODIC_TSV, "a", newline="") as f:
        w = csv.writer(f, delimiter="\t")
        if new_file:
            w.writerow(["head", "relation", "tail"])
        w.writerows(patterns)
    print(f"  [TSV] {len(patterns)} patterns written")

    for h, r, t in patterns:
        write_episodic(h, r, t, agg.get("chosen_agent","?"),
                       f"grounded: {h} --{r}--> {t}")
    write_semantic(agg.get("chosen_agent","?"), record.get("relation","?"),
                   list({r for _, r, _ in patterns}))

In [52]:
def run_pipeline(record, tsv_memory=None, df_ref=None, constraints=None):
    if "true_tail" not in record:
        record["true_tail"] = record.get("tail", "")
    if "hop_type" not in record:
        record["hop_type"] = "multi"
    t = f"({record['head']}, {record['relation']}, {record['true_tail']})"
    print(f"\n{'='*55}\nPIPELINE  {t}\n"
          f"rank={record['true_rank']}  hop={record['hop_type']}\n{'='*55}")
    a_out, b_out = run_parallel_staggered(record, tsv_memory=tsv_memory)
    s_a = grounded_score(a_out, record, "A", df_ref=df_ref, constraints=constraints)
    s_b = grounded_score(b_out, record, "B",                constraints=constraints)
    agg = aggregate(a_out, b_out, s_a, s_b, record)
    writeback(agg, record, constraints=constraints)
    correct = (agg.get("final_answer","").strip().lower()
               == record["true_tail"].strip().lower())
    print(f"\n{'✓' if correct else '✗'}  final={agg.get('final_answer')}  "
          f"agent={agg.get('chosen_agent')}")
    return {
        "triple":        t,
        "true_tail":     record["true_tail"],
        "hop_type":      record["hop_type"],
        "model_rank":    record["true_rank"],
        "agent_a":       a_out,
        "agent_b":       b_out,
        "score_a":       s_a,
        "score_b":       s_b,
        "aggregator":    agg,
        "final_correct": correct,
    }

In [53]:
# Run this to see what training data has for Lara Fabian
key = ("Lara Fabian", "occupation")
print(f"Known tails: {ent_rel_to_tails.get(key, [])}")

# Also check what candidates look like for this record
lara = next(r for r in val_hard if r["head"] == "Lara Fabian")
print(f"Top-15 candidates:")
for i, (e, s) in enumerate(lara["top5"]):
    print(f"  rank {i+1}: {e}")
print(f"True tail: {lara['tail']}  at rank {lara['true_rank']}")

Known tails: ['singer', 'musician', 'singer-songwriter']
Top-15 candidates:
  rank 1: musician
  rank 2: singer
  rank 3: singer-songwriter
  rank 4: actor
  rank 5: pianist
  rank 6: composer
  rank 7: recording artist
  rank 8: record producer
  rank 9: guitarist
  rank 10: songwriter
  rank 11: opera singer
  rank 12: philanthropist
  rank 13: film actor
  rank 14: poet
  rank 15: writer
True tail: recording artist  at rank 7


In [54]:

CHECKPOINT_FILE = "/kaggle/working/val_hard_checkpoint1.json"
RESULTS_FILE    = "/kaggle/working/val_hard_results.json"

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f: return json.load(f)
    return {}

def save_checkpoint(cp):
    with open(CHECKPOINT_FILE, "w") as f: json.dump(cp, f, indent=2)

def triple_key(record):
    return f"{record['head']}|{record['relation']}|{record['tail']}"

checkpoint   = load_checkpoint()
already_done = set(checkpoint.keys())
remaining    = [r for r in val_hard if triple_key(r) not in already_done]

print(f"val_hard total:    {len(val_hard)}")
print(f"Already completed: {len(already_done)}")
print(f"Remaining:         {len(remaining)}")

all_results = list(checkpoint.values())

for i, record in enumerate(remaining):
    key = triple_key(record)
    print(f"\n[{i+1}/{len(remaining)}]  {key}  rank={record['true_rank']}")
    try:
        result = run_pipeline(record, tsv_memory=tsv_memory, df_ref=df_train, constraints=constraints)
        checkpoint[key] = result
        all_results.append(result)
        save_checkpoint(checkpoint)
    except Exception as e:
        print(f"  [ERROR] {key}: {e}")
        checkpoint[key] = {"error": str(e), "triple": key}
        save_checkpoint(checkpoint)
    if i < len(remaining) - 1:
        time.sleep(BETWEEN_RECORDS)


clean_results = [r for r in all_results if "error" not in r]
with open(RESULTS_FILE, "w") as f: json.dump(clean_results, f, indent=2)

errors  = [r for r in all_results if "error" in r]
correct = [r for r in clean_results if r.get("final_correct")]
agents  = [r["aggregator"].get("chosen_agent") for r in clean_results]
ftypes  = [r["aggregator"].get("failure_type")  for r in clean_results]

print(f"\n{'='*55}  FINAL SUMMARY")
print(f"Total processed:  {len(all_results)}")
print(f"Errors/skipped:   {len(errors)}")
print(f"Clean results:    {len(clean_results)}")
print(f"Correct:          {len(correct)} / {len(clean_results)}"
      f"  ({100*len(correct)/max(len(clean_results),1):.1f}%)")
print(f"\nAgent chosen:")
for agent, count in Counter(agents).most_common():
    print(f"  Agent {agent}: {count}")
print(f"\nFailure types:")
for ftype, count in Counter(ftypes).most_common():
    print(f"  {ftype}: {count}")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


val_hard total:    101
Already completed: 0
Remaining:         101

[1/101]  Wassily Kandinsky|country of citizenship|Soviet Union  rank=4

PIPELINE  (Wassily Kandinsky, country of citizenship, Soviet Union)
rank=4  hop=multi
  [A] (Wassily Kandinsky, country of citizenship, Soviet Union)
  [B] (Wassily Kandinsky, country of citizenship, Soviet Union)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [A] pred=Germany  conf=0.85


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


[101/101]  Honduras|part of|Hispanic America  rank=2

PIPELINE  (Honduras, part of, Hispanic America)
rank=2  hop=multi
  [A] (Honduras, part of, Hispanic America)
  [B] (Honduras, part of, Hispanic America)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [A] pred=Hispanic America  conf=0.95


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [B] pred=Hispanic America  conf=0.95
  [verify A] all relations confirmed
  [subgraph verify A] ✓ all relations grounded in subgraph
  [score A] coverage=0.00  grounding=1.00  type=0.16  path=0.00  calib=0.95  → quality=0.435  correct=True
  [verify B] all relations confirmed
  [subgraph verify B] ✓ all relations grounded in subgraph
  [score B] coverage=0.00  grounding=1.00  type=0.16  path=1.00  calib=0.95  → quality=0.527  correct=True
  [Route] agent B  quality=0.527  contam=0.0
  [Route] B quality (0.53) > threshold, A quality (0.43) did not
  [Agg] LLM labelling …
  [Agg] final=Hispanic America  agent=B  type=resolved
  [TSV] 1 patterns written

✓  final=Hispanic America  agent=B

=======================================================  FINAL SUMMARY
Total processed:  101
Errors/skipped:   0
Clean results:    101
Correct:          84 / 101  (83.2%)

Agent chosen:
  Agent B: 101

Failure types:
  resolved: 89
  similarity_confusion: 10
  both_failed: 1
  structural_gap: 1
